# MLPerf-*inspired* Inference — ResNet-50 / ImageNet (Vision) on Colab

Runs the MLPerf `vision/classification_and_detection` reference harness (ResNet-50, PyTorch backend)
under LoadGen on a Colab GPU: **top-1 accuracy + Offline performance**.

> **Not a conformant MLPerf result** — short config + subset data, for quick cross-hardware
> comparison, not submission. (This is a pre-hardening execution snapshot — see the source
> `mlperf_resnet50_colab.ipynb` for the pinned/checksum-verified version.)

Dataset = **Imagenette** (fast.ai's ungated real-ImageNet val subset, 10 classes) since
ImageNet-1k val is access-gated. Expected top-1 ≈ **84–85%** (full 1000-way classifier
restricted to these images).

> Set **Runtime → Change runtime type → GPU** first.

## 1 · GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),
      '|',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

name, memory.total [MiB]
Tesla T4, 15360 MiB


torch 2.11.0+cu128 | cuda True | Tesla T4


## 2 · Dependencies

torch/torchvision/cv2 are preinstalled on Colab; add loadgen + pycocotools (main.py imports coco).

In [ ]:
!pip -q install mlcommons-loadgen pycocotools opencv-python-headless
import mlperf_loadgen, pycocotools, cv2; print('loadgen + pycocotools + cv2 ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.2/472.2 kB 24.8 MB/s eta 0:00:00


loadgen + pycocotools + cv2 ready


## 3 · Clone the MLPerf inference repo

In [ ]:
import os
if not os.path.isdir('/content/inference'):
    !git clone --depth 1 https://github.com/mlcommons/inference.git /content/inference
%cd /content/inference/vision/classification_and_detection
!git -C /content/inference rev-parse --short HEAD

Cloning into '/content/inference'...


remote: Enumerating objects: 1467, done.
remote: Counting objects: 100% (1467/1467), done.


remote: Compressing objects: 100% (1120/1120), done.


remote: Total 1467 (delta 325), reused 983 (delta 233), pack-reused 0 (from 0)
Receiving objects: 100% (1467/1467), 261.35 MiB | 19.72 MiB/s, done.


Resolving deltas: 100% (325/325), done.


Updating files: 100% (1371/1371), done.


/content/inference/vision/classification_and_detection


da738a5


## 4 · Model + Imagenette dataset

In [ ]:
%%bash
mkdir -p /content/vision && cd /content/vision
[ -s resnet50-19c8e357.pth ] || wget -q -O resnet50-19c8e357.pth 'https://zenodo.org/record/4588417/files/resnet50-19c8e357.pth?download=1'
if [ ! -d imagenette2-320 ]; then wget -q -O i.tgz https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz && tar xzf i.tgz && rm i.tgz; fi
echo 'val images:' $(find imagenette2-320/val -iname '*.JPEG' | wc -l)

val images: 3925


## 5 · Build full model + val_map + patch backend

The Zenodo `.pth` is a legacy-format **state_dict** → build a full torchvision model
(`weights_only=False`). The `pytorch-native` backend returns a bare GPU tensor, which the
post-process can't consume → patch it to return a numpy batch. Build `val_map.txt` with the
10 Imagenette synset → torchvision class-index labels.

In [ ]:
import torch, torchvision, os, glob
# full model
sd = torch.load('/content/vision/resnet50-19c8e357.pth', map_location='cpu', weights_only=False)
m = torchvision.models.resnet50(); m.load_state_dict(sd); m.eval()
torch.save(m, '/content/vision/resnet50_full.pth'); print('saved resnet50_full.pth')
# val_map
SYN={'n01440764':0,'n02102040':217,'n02979186':482,'n03000684':491,'n03028079':497,
     'n03394916':566,'n03417042':569,'n03425413':571,'n03445777':574,'n03888257':701}
vd='/content/vision/imagenette2-320/val'; L=[]
for s,i in SYN.items():
    for p in sorted(glob.glob(os.path.join(vd,s,'*.JPEG'))): L.append(f'{os.path.relpath(p,vd)} {i}')
open(os.path.join(vd,'val_map.txt'),'w').write('\n'.join(L)+'\n'); print('val_map entries:',len(L))
# patch backend_pytorch_native to return [numpy batch]
f='python/backend_pytorch_native.py'; s=open(f).read()
old='        with torch.no_grad():\n            output = self.model(feed[key])\n        return output'
new='        with torch.no_grad():\n            output = self.model(feed[key])\n        return [output.cpu().numpy()]'
open(f,'w').write(s.replace(old,new)); print('backend patched:', old.split(chr(10))[-1].strip(),'->','return [output.cpu().numpy()]')

saved resnet50_full.pth
val_map entries: 3925
backend patched: return output -> return [output.cpu().numpy()]


## 6 · Accuracy (top-1)

`main.py` accuracy mode crashes in its post-test percentile stats (numpy2 bug) *after*
LoadGen writes the accuracy log → tolerate it, then score with `tools/accuracy-imagenet.py`.

In [ ]:
%%bash
cat > /content/vision/demo.conf <<'CONF'
resnet50.Offline.target_qps = 700
resnet50.Offline.min_duration = 10000
resnet50.Offline.min_query_count = 1
CONF
cd python
python main.py --profile resnet50-pytorch --backend pytorch-native \
  --model /content/vision/resnet50_full.pth --dataset-path /content/vision/imagenette2-320/val \
  --user_conf /content/vision/demo.conf --scenario Offline --accuracy \
  --output /content/vision/out_acc >/tmp/acc.log 2>&1 || true
echo 'acc log bytes:' $(stat -c%s /content/vision/out_acc/mlperf_log_accuracy.json)
python ../tools/accuracy-imagenet.py --mlperf-accuracy-file /content/vision/out_acc/mlperf_log_accuracy.json \
  --imagenet-val-file /content/vision/imagenette2-320/val/val_map.txt --dtype float32 2>&1 | tail -2

acc log bytes: 233283
accuracy=84.535%, good=3318, total=3925


## 7 · Performance (Offline, VALID)

In [ ]:
%%bash
cd python
python main.py --profile resnet50-pytorch --backend pytorch-native \
  --model /content/vision/resnet50_full.pth --dataset-path /content/vision/imagenette2-320/val \
  --user_conf /content/vision/demo.conf --scenario Offline \
  --output /content/vision/out_perf >/tmp/perf.log 2>&1 || true
sed -n '1,12p' /content/vision/out_perf/mlperf_log_summary.txt

MLPerf Results Summary
SUT name : PySUT
Scenario : Offline
Mode     : PerformanceOnly
Samples per second: 304.373
Result is : VALID
  Min duration satisfied : Yes
  Min queries satisfied : Yes
  Early stopping satisfied: Yes



## Done ✅ — MLPerf-inspired ResNet-50 vision run on Colab GPU (top-1 + LoadGen-VALID perf under a short config).